A decorator is a callable that takes a function (or class) and returns a replacement. The `@decorator` syntax is shorthand for reassigning the name: `func = decorator(func)`. Decorators let you add behaviour — logging, timing, caching, validation, retry logic — to existing functions without touching their source code. Because Python treats functions as first-class objects and supports closures, the entire mechanism is a natural consequence of the language itself. This notebook builds decorators from first principles, covers `functools.wraps`, decorator factories, stacking, class-based decorators, and a set of practical patterns you will reach for repeatedly.

## Functions Are First-Class Objects

Everything that makes decorators possible starts with one fact: functions are objects. They can be assigned to variables, stored in data structures, passed as arguments, and returned from other functions.

In [ ]:
def greet(name):
    return f"Hello, {name}!"


# Assign to a variable — both names refer to the same function object
say_hello = greet
print(say_hello("Alice"))       # Hello, Alice!
print(greet is say_hello)       # True

# Pass as an argument
def apply(func, value):
    return func(value)

print(apply(greet, "Bob"))      # Hello, Bob!

# Return from a function
def make_adder(n):
    def adder(x):
        return x + n
    return adder

add5 = make_adder(5)
print(add5(10))                 # 15
print(add5(20))                 # 25

## Closures — The Mechanism Behind Decorators

A **closure** is a nested function that remembers the variables from the enclosing scope even after that scope has returned. This is how wrappers keep hold of the original function.

In [ ]:
def make_counter(start=0):
    count = start                   # enclosed variable

    def increment(step=1):
        nonlocal count
        count += step
        return count

    return increment


counter = make_counter()
print(counter())    # 1
print(counter())    # 2
print(counter(5))   # 7

# Each call to make_counter creates an independent closure
other = make_counter(100)
print(other())      # 101
print(counter())    # 8 — unaffected

# The closed-over variables are stored in __closure__
print(counter.__closure__[0].cell_contents)   # 8

## Building a Decorator from Scratch

A decorator is a function that accepts a function and returns a new function. Typically the returned function is a wrapper: it runs some extra code, calls the original, and returns the result.

In [ ]:
import time

def timer(func):
    """Decorator: print how long func takes to run."""
    def wrapper(*args, **kwargs):
        start  = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.6f}s")
        return result
    return wrapper


# Manual application — func = decorator(func)
def slow_sum(n):
    return sum(range(n))

slow_sum = timer(slow_sum)    # decorate manually
print(slow_sum(1_000_000))

## The `@` Syntax

`@decorator` placed on the line immediately before a `def` is exactly equivalent to `func = decorator(func)`. It is just syntactic sugar — cleaner and more visible.

In [ ]:
@timer
def fast_sum(n):
    return sum(range(n))


# Equivalent to: fast_sum = timer(fast_sum)
print(fast_sum(1_000_000))
print(fast_sum(10_000_000))

# The wrapper is what 'fast_sum' now points to
print(fast_sum.__name__)    # wrapper  ← problem: identity lost

## `functools.wraps` — Preserving Identity

The wrapper hides the original function's `__name__`, `__doc__`, `__module__`, and other attributes. `functools.wraps` copies them from the wrapped function to the wrapper. **Always use it.**

In [ ]:
import functools


def timer(func):
    @functools.wraps(func)        # copies __name__, __doc__, __qualname__, etc.
    def wrapper(*args, **kwargs):
        start   = time.perf_counter()
        result  = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.6f}s")
        return result
    return wrapper


@timer
def slow_sum(n):
    """Return the sum of 0..n-1."""
    return sum(range(n))


print(slow_sum.__name__)    # slow_sum  ✓
print(slow_sum.__doc__)     # Return the sum of 0..n-1.  ✓
print(slow_sum(500_000))

# __wrapped__ lets you access the original function
print(slow_sum.__wrapped__)   # <function slow_sum at ...>

## Decorator Factories (Decorators with Arguments)

To pass arguments to a decorator, add one more layer: a **factory function** that accepts the arguments and returns the actual decorator.

In [ ]:
def repeat(n):
    """Decorator factory: run the decorated function n times."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator


@repeat(3)
def say(msg):
    print(msg)


# Equivalent to: say = repeat(3)(say)
say("hello")    # printed three times

In [ ]:
def retry(times=3, exceptions=(Exception,), delay=0.0):
    """Retry the decorated function up to `times` times on failure."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    last_error = e
                    print(f"  Attempt {attempt} failed: {e}")
                    if delay and attempt < times:
                        time.sleep(delay)
            raise last_error
        return wrapper
    return decorator


call_count = 0

@retry(times=4, exceptions=(ValueError,))
def flaky():
    global call_count
    call_count += 1
    if call_count < 3:
        raise ValueError(f"not ready yet (call {call_count})")
    return f"success on call {call_count}"


print(flaky())

## Stacking Decorators

Multiple `@decorator` lines stack from bottom to top: the decorator closest to the function is applied first, then the one above it, and so on.

In [ ]:
def bold(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f"<b>{func(*args, **kwargs)}</b>"
    return wrapper


def italic(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f"<i>{func(*args, **kwargs)}</i>"
    return wrapper


@bold          # applied second  →  bold(italic(greet))
@italic        # applied first   →  italic(greet)
def greet(name):
    return f"Hello, {name}!"


print(greet("Alice"))     # <b><i>Hello, Alice!</i></b>

# Compare: reversed order
@italic
@bold
def greet2(name):
    return f"Hello, {name}!"

print(greet2("Alice"))    # <i><b>Hello, Alice!</b></i>

## Class-Based Decorators

A decorator can be any callable. A class with `__call__` defined is callable — making it a natural choice when the decorator needs to maintain state across calls.

In [ ]:
class CallCounter:
    """Decorator that counts how many times the wrapped function is called."""

    def __init__(self, func):
        functools.update_wrapper(self, func)   # equivalent to @functools.wraps
        self.func  = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        return self.func(*args, **kwargs)


@CallCounter
def add(a, b):
    return a + b


add(1, 2)
add(3, 4)
add(5, 6)

print(add(10, 20))          # 30
print(f"Called {add.count} times")   # Called 4 times

In [ ]:
class RateLimit:
    """Allow at most `max_calls` calls per `period` seconds."""

    def __init__(self, max_calls, period=1.0):
        self.max_calls = max_calls
        self.period    = period
        self.calls     = []

    def __call__(self, func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            now = time.monotonic()
            # purge calls outside the current window
            self.calls = [t for t in self.calls if now - t < self.period]
            if len(self.calls) >= self.max_calls:
                raise RuntimeError(
                    f"Rate limit: max {self.max_calls} calls per {self.period}s"
                )
            self.calls.append(now)
            return func(*args, **kwargs)
        return wrapper


@RateLimit(max_calls=3, period=1.0)
def api_call(endpoint):
    return f"response from {endpoint}"


for i in range(3):
    print(api_call(f"/data/{i}"))

try:
    api_call("/data/4")          # 4th call within the same second
except RuntimeError as e:
    print(e)

## Decorating Classes

Decorators work on classes too. The decorator receives the class object, can modify or replace it, and returns the result. This is a lighter-weight alternative to metaclasses for class-level transformations.

In [ ]:
def singleton(cls):
    """Ensure only one instance of cls is ever created."""
    instances = {}

    @functools.wraps(cls)
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]

    return get_instance


@singleton
class Config:
    def __init__(self, env="production"):
        self.env = env


a = Config()
b = Config("staging")   # ignored — first instance already cached

print(a is b)           # True
print(a.env)            # production

In [ ]:
def add_repr(cls):
    """Auto-generate __repr__ from the instance __dict__."""
    def __repr__(self):
        attrs = ", ".join(f"{k}={v!r}" for k, v in self.__dict__.items())
        return f"{cls.__name__}({attrs})"
    cls.__repr__ = __repr__
    return cls


@add_repr
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y


print(Point(3, 4))    # Point(x=3, y=4)

## `functools.cache` and `functools.lru_cache`

The standard library ships two memoisation decorators that cache the results of function calls, keyed by their arguments.

- `@functools.cache` — unbounded cache (Python 3.9+). Equivalent to `lru_cache(maxsize=None)`.
- `@functools.lru_cache(maxsize=N)` — keeps only the `N` most recently used entries.

In [ ]:
@functools.cache
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)


print(fib(50))   # 12586269025 — instant; without caching this would be 2^50 calls
print(fib.cache_info())   # hits, misses, maxsize, currsize


# LRU cache with a bounded size
@functools.lru_cache(maxsize=128)
def expensive(n):
    time.sleep(0.001)      # simulate work
    return n ** 2


for i in [1, 2, 3, 1, 2, 3]:   # second pass hits cache
    expensive(i)

info = expensive.cache_info()
print(f"hits={info.hits}, misses={info.misses}")

## Practical Patterns

A small set of decorator patterns covers the majority of real-world use cases.

In [ ]:
import logging

logging.basicConfig(level=logging.DEBUG, format="%(levelname)s %(message)s")


def logged(func):
    """Log entry, exit, and any exception for the decorated function."""
    logger = logging.getLogger(func.__module__)

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        logger.debug("Calling %s(%s)", func.__name__,
                     ", ".join(map(repr, args)))
        try:
            result = func(*args, **kwargs)
            logger.debug("%s → %r", func.__name__, result)
            return result
        except Exception as e:
            logger.error("%s raised %s: %s", func.__name__, type(e).__name__, e)
            raise
    return wrapper


@logged
def divide(a, b):
    return a / b


print(divide(10, 2))

try:
    divide(1, 0)
except ZeroDivisionError:
    pass

In [ ]:
def validate(**types):
    """Validate argument types at call time.

    Usage:  @validate(x=int, y=(int, float))
    """
    def decorator(func):
        import inspect
        sig = inspect.signature(func)
        params = list(sig.parameters)

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            bound = sig.bind(*args, **kwargs)
            bound.apply_defaults()
            for name, expected in types.items():
                if name in bound.arguments:
                    value = bound.arguments[name]
                    if not isinstance(value, expected):
                        raise TypeError(
                            f"{func.__name__}: argument '{name}' must be "
                            f"{expected}, got {type(value).__name__}"
                        )
            return func(*args, **kwargs)
        return wrapper
    return decorator


@validate(x=int, y=(int, float))
def scale(x, y):
    return x * y


print(scale(3, 2.5))       # 7.5

try:
    scale("3", 2)          # x is a str, not int
except TypeError as e:
    print(e)

In [ ]:
# Decorator that measures and records execution times
import statistics


def benchmark(func):
    """Record each call's duration; expose .stats() on the wrapper."""
    durations = []

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        durations.append(time.perf_counter() - start)
        return result

    def stats():
        if not durations:
            return "No calls recorded."
        return {
            "calls":  len(durations),
            "mean":   statistics.mean(durations),
            "median": statistics.median(durations),
            "min":    min(durations),
            "max":    max(durations),
        }

    wrapper.stats = stats
    return wrapper


@benchmark
def sort_random(n):
    import random
    return sorted(random.random() for _ in range(n))


for _ in range(5):
    sort_random(100_000)

s = sort_random.stats()
print(f"calls={s['calls']}, mean={s['mean']:.4f}s, min={s['min']:.4f}s")

## Decorators that Work With or Without Arguments

A common usability improvement: make the decorator work both as `@dec` and `@dec(arg=value)` by checking whether the first argument is a callable.

In [ ]:
def repeat(_func=None, *, times=2):
    """Repeat the decorated function.

    Works as @repeat (uses default times=2)
    or as @repeat(times=5) (explicit).
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper

    if _func is not None:      # called as @repeat — no parentheses
        return decorator(_func)
    return decorator           # called as @repeat(...) — return the decorator


@repeat                    # no parentheses — uses times=2
def ping():
    print("ping")

ping()
print()

@repeat(times=4)           # explicit argument
def pong():
    print("pong")

pong()

## Summary

- A **decorator** is a callable that takes a function (or class) and returns a replacement. `@dec` is shorthand for `func = dec(func)`.
- Functions are first-class: assignable, passable, returnable. **Closures** let inner functions remember variables from outer scopes — the mechanism that makes wrappers hold onto the original function.
- A decorator typically defines a `wrapper(*args, **kwargs)` inner function, runs extra logic, calls the original via `func(*args, **kwargs)`, and returns the result.
- **Always apply `@functools.wraps(func)`** to the wrapper. It copies `__name__`, `__doc__`, `__qualname__`, and sets `__wrapped__` for introspection.
- **Decorator factories** add one more nesting level: a function that accepts arguments and returns the actual decorator. `@factory(arg)` is equivalent to `func = factory(arg)(func)`.
- **Stacking** applies decorators bottom-up: the closest decorator to the `def` is applied first.
- **Class-based decorators** use `__call__` and are natural when the decorator needs to maintain state. `functools.update_wrapper(self, func)` replaces `@functools.wraps` when you cannot use the decorator syntax.
- **Class decorators** receive the class object and can modify or replace it — lighter than metaclasses for most transformations.
- **`@functools.cache`** (unbounded) and **`@functools.lru_cache(maxsize=N)`** (bounded) provide ready-made memoisation.
- Common patterns: `timer`, `retry`, `logged`, `validate`, `benchmark`, `singleton`, `rate_limit`.
- To make a decorator work with and without parentheses, check `if _func is not None` using a keyword-only sentinel parameter.